# Clustering calibration: what identity does the MinHash threshold actually catch?

**Question answered.** At J&ge;0.90 over 7-mers, what nucleotide identity is actually being
joined into one cluster? Is that threshold catching the near-duplicates that matter (same
transmission pair, same patient at two timepoints, ~97-99% identity)? This notebook (a)
re-derives clusters with **canonical, step-1, strand-independent k-mers** instead of the
codon-stepped sketch, (b) sweeps the join threshold across the identities that matter
biologically, (c) cross-checks against CD-HIT, and (d) reports macro-F1 under each
threshold's cluster-disjoint split.

**GPU needed.** No -- this notebook only needs the classical models (Linear SVM) to report
macro-F1 per threshold; CPU is enough. If you also want the CNNs re-run per threshold, add
GPU, but that is not required to answer ClusterCalib's core question.

**Runtime.** Building MinHash sketches for ~24k sequences at k=7, step 1: a few minutes with
`datasketch`. The threshold sweep (5 thresholds) plus the classical-model macro-F1 at each:
another 5-10 minutes. CD-HIT clustering (installed via conda/pip inside this notebook, no
manual download) on ~24k pol sequences: a few minutes per identity level. Whole notebook
should finish in under an hour.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install datasketch / CD-HIT (both installed here, not downloaded by you)

In [ ]:

import warnings; warnings.filterwarnings("ignore")
!pip install -q datasketch biopython
# CD-HIT via bioconda -- if conda is unavailable in this Kaggle image, this cell falls back
# to compiling from source (a few minutes) so the notebook still runs end to end.
import subprocess, shutil, os
if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    r = subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


In [ ]:

import numpy as np, pandas as pd, math
import matplotlib.pyplot as plt
from datasketch import MinHash
from Bio.Seq import Seq
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, balanced_accuracy_score, adjusted_rand_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)


### 2. Load data

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
print(f"{len(df)} sequences | {n_cls} classes")


### 3. Canonical, step-1 k-mer MinHash sketches (fixes the codon-stepping bug)

Codon-stepped (step 3) sketches are reading-frame dependent: two sequences offset by 1 nt produce almost disjoint k-mer sets for no biological reason. This uses step-1 canonical k-mers (min of k-mer and its reverse complement), which is strand- and frame-independent.

In [ ]:

K = 7
NUM_PERM = 128   # larger sketch than the original 64-permutation one, for a more accurate Jaccard estimate

def canonical_kmers(seq, k=K):
    seq = seq.upper().replace("U", "T")
    rc = str(Seq(seq).reverse_complement())
    n = len(seq)
    for i in range(n - k + 1):
        f = seq[i:i+k]
        if "N" in f or any(c not in "ACGT" for c in f):
            continue
        r = rc[n-k-i : n-i]
        yield min(f, r)

def build_minhash(seq):
    m = MinHash(num_perm=NUM_PERM)
    for km in canonical_kmers(seq):
        m.update(km.encode("utf8"))
    return m

print("Building MinHash sketches for all sequences (a few minutes)...")
sketches = [build_minhash(s) for s in X_all]
print("done:", len(sketches))


### 4. Union-find clustering at each threshold, biology-anchored

In [ ]:

def jaccard_to_identity(J, k=K):
    if J <= 0:
        return 0.0
    D = -1.0/k * math.log((2*J) / (1+J))
    return 1 - D

# thresholds chosen from the worked example in the reviewer's note (~99.2%, 98%, 97%, 95%, 93% identity)
THRESHOLDS = [0.90, 0.78, 0.68, 0.54, 0.44]

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

def cluster_at_threshold(sketches, thresh, lsh_bands=None):
    from datasketch import MinHashLSH
    lsh = MinHashLSH(threshold=thresh, num_perm=NUM_PERM)
    for i, m in enumerate(sketches):
        lsh.insert(str(i), m)
    uf = UnionFind(len(sketches))
    seen = set()
    for i, m in enumerate(sketches):
        if i in seen:
            continue
        for j_str in lsh.query(m):
            j = int(j_str)
            if j != i:
                uf.union(i, j)
        seen.add(i)
    roots = [uf.find(i) for i in range(len(sketches))]
    remap = {r: k for k, r in enumerate(sorted(set(roots)))}
    return np.array([remap[r] for r in roots])

cluster_results = {}
for t in THRESHOLDS:
    approx_identity = jaccard_to_identity(t)
    labels = cluster_at_threshold(sketches, t)
    n_clusters = len(set(labels))
    _, counts = np.unique(labels, return_counts=True)
    largest = counts.max()
    cluster_results[t] = dict(identity=approx_identity, labels=labels, n_clusters=n_clusters, largest=largest)
    print(f"J>={t:.2f} (~{approx_identity*100:.1f}% identity): {n_clusters} clusters, largest={largest}")


### 5. Cross-check against CD-HIT at 95% and 97% identity

In [ ]:

def write_fasta(path, seqs, ids):
    with open(path, "w") as f:
        for i, s in zip(ids, seqs):
            f.write(f">{i}\n{s}\n")

write_fasta(f"{OUT}/pol.fasta", X_all, range(len(X_all)))

def run_cdhit(identity, word_size):
    out = f"{OUT}/pol_cdhit_{int(identity*100)}"
    cmd = f"cd-hit-est -i {OUT}/pol.fasta -o {out} -c {identity} -n {word_size} -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out, r.returncode

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1)
    cluster_id = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cluster_id += 1
            else:
                seq_id = int(line.split(">")[1].split("...")[0])
                labels[seq_id] = cluster_id
    return labels

cdhit_labels = {}
if shutil.which("cd-hit-est") is not None:
    for identity, word_size in [(0.95, 10), (0.97, 10)]:
        out, rc = run_cdhit(identity, word_size)
        if rc == 0 and os.path.exists(out + ".clstr"):
            labs = parse_cdhit_clusters(out + ".clstr", len(X_all))
            cdhit_labels[identity] = labs
            print(identity, "CD-HIT clusters:", len(set(labs)))
        else:
            print(f"CD-HIT run at {identity} failed (returncode={rc}); skipping cross-check at this level.")
else:
    print("cd-hit-est unavailable in this environment -- skipping the CD-HIT cross-check; "
          "MinHash-only thresholds above are still valid and are the main ClusterCalib result.")


### 6. Agreement between MinHash and CD-HIT (Adjusted Rand Index)

In [ ]:

ari_rows = []
for j_thresh, res in cluster_results.items():
    for identity, cdhit_lab in cdhit_labels.items():
        if abs(res["identity"] - identity) < 0.03:  # compare roughly matching identity levels
            ari = adjusted_rand_score(cdhit_lab, res["labels"])
            ari_rows.append(dict(minhash_J=j_thresh, minhash_identity=res["identity"],
                                  cdhit_identity=identity, adjusted_rand_index=ari))
            print(f"MinHash J>={j_thresh} (~{res['identity']*100:.1f}%) vs CD-HIT {identity*100:.0f}%: ARI={ari:.3f}")
pd.DataFrame(ari_rows).to_csv(f"{OUT}/clustercalib_minhash_vs_cdhit_ari.csv", index=False)


### 7. macro-F1 under each clustering threshold (cluster-disjoint split, Linear SVM, 3 seeds)

In [ ]:
def grouped_split(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    return trv, te

DEGENERATE_FRAC = 0.30  # same threshold as the Section 8 warning below

def is_degenerate(res, n_total):
    return (res["largest"] / n_total) > DEGENERATE_FRAC

def macro_f1_for_labels(cluster_labels, seeds=(0, 1, 2)):
    """Returns (mean, std, n_valid_seeds, note). Guards against the degenerate-clustering
    failure mode: when one cluster dominates the data, StratifiedGroupKFold can hand back an
    empty or near-empty test fold, or a test fold missing entire classes. We catch that per
    seed rather than letting the whole sweep crash, and report how many of the seeds actually
    produced a usable split."""
    f1s = []
    notes = []
    for seed in seeds:
        tr, te = grouped_split(X_all, y_all, cluster_labels, seed)
        if len(te) == 0 or len(tr) == 0:
            notes.append(f"seed {seed}: empty split (0 train or 0 test)")
            continue
        if len(np.unique(y_all[te])) < 2:
            notes.append(f"seed {seed}: test fold has <2 classes present")
            continue
        try:
            vec = TfidfVectorizer(analyzer="char", ngram_range=(6, 6), max_features=512, lowercase=False)
            Xtr = vec.fit_transform(X_all[tr]); Xte = vec.transform(X_all[te])
            clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
            clf.fit(Xtr, y_all[tr])
            pred = clf.predict(Xte)
            f1s.append(f1_score(y_all[te], pred, average="macro"))
        except ValueError as e:
            notes.append(f"seed {seed}: {e}")
    if len(f1s) == 0:
        return float("nan"), float("nan"), 0, "; ".join(notes) if notes else "no valid seeds"
    return float(np.mean(f1s)), float(np.std(f1s)), len(f1s), "; ".join(notes)

rows = []
for t, res in cluster_results.items():
    degenerate_flag = is_degenerate(res, len(X_all))
    if degenerate_flag:
        print(f"J>={t:.2f}: largest cluster is {res['largest']/len(X_all)*100:.1f}% of the data "
              f"-- attempting the split anyway to see how it fails, but treat this row as degenerate.")
    f1_mean, f1_std, n_valid, note = macro_f1_for_labels(res["labels"])
    rows.append(dict(jaccard_threshold=t, approx_identity=res["identity"],
                      n_clusters=res["n_clusters"], largest_cluster=res["largest"],
                      largest_cluster_frac=round(res["largest"]/len(X_all), 4),
                      degenerate=degenerate_flag,
                      macro_f1_mean=round(f1_mean, 4) if n_valid else np.nan,
                      macro_f1_std=round(f1_std, 4) if n_valid else np.nan,
                      n_valid_seeds=n_valid, note=note))
    print(rows[-1])

clustercalib_table = pd.DataFrame(rows).sort_values("jaccard_threshold")
clustercalib_table.to_csv(f"{OUT}/clustercalib_clustering_calibration_table.csv", index=False)
clustercalib_table

### 8. Watch for degenerate clustering (e.g. subtype B collapsing to one giant cluster)

In [ ]:

for t, res in cluster_results.items():
    labels = res["labels"]
    frac_largest = res["largest"] / len(labels)
    flag = "  <-- WARNING: dominant single cluster, split will be degenerate" if frac_largest > 0.3 else ""
    print(f"J>={t:.2f} (~{res['identity']*100:.1f}%): largest cluster = {res['largest']} "
          f"({frac_largest*100:.1f}% of data){flag}")


### 9. Recommendation cell — fill in after inspecting the table above

In [ ]:

# Per the reviewer: run the main analysis at ~97% identity (J~=0.68) and report 95% (J~=0.54)
# as a sensitivity analysis, unless the table/warnings above say otherwise (e.g. subtype B
# collapsing into one cluster at 95%, in which case back off to 97% only).
print(clustercalib_table)


### 10. Zip outputs

In [ ]:
!cd /kaggle/working && zip -r clustercalib_clustering_calibration_outputs.zip clustercalib_clustering_calibration_table.csv clustercalib_minhash_vs_cdhit_ari.csv